In [2]:
system("conda install r-coda r-dotcall64 r-statnet.common r-network r-spam r-maps r-wk r-sna r-vegan r-corpcor r-fields r-permute r-fastmatch r-rbibutils r-classint r-s2 r-units r-ape r-apcluster r-bipartite r-cluster r-dbscan r-dynamictreecut r-fastcluster r-igraph r-mathjaxr r-phangorn r-rdpack r-segmented r-sf")

In [3]:
install.packages("fastkmedoids")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [4]:
install.packages("bioregion")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [2]:
install.packages("devtools")
devtools::install_github("Farewe/biogeonetworks")

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done




── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpA8BWxT/remotesd16d2d480594/Farewe-biogeonetworks-caea7c6/DESCRIPTION’ ... OK
* preparing ‘biogeonetworks’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Omitted ‘LazyData’ from DESCRIPTION
* building ‘biogeonetworks_0.1.2.tar.gz’



In [3]:
##Load packages
library("bioregion")
library("biogeonetworks")

In [4]:
##Parameters
rm(list = ls())
#Input file 
file_sp <- "galaxy_inputs/bioregion/mod07-2024_occurrences_verified_wild_2024-06-09.tabular.tabular" #c("D:/Documents/Pro/Thesis/Cloud/Data/data_ind_samp_loc.xlsx)
col_sp <- "species"
col_loc <- "ID_sample"
rm_sp <- "ilyocryptus_spinifer"

#List columns to add in output file
l_col <- c("x_longitude", "y_latitude", "sampling_date_DD_MM_YYYY", "sampling_year", "ID_locality")
  
#Output directory
out_dir <- "outputs/collection"

In [6]:
library(dplyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [1]:
system("conda install python-infomap=2.8.0")

In [2]:
install_binaries(binpath = "tempdir" ,
                 download_only =  FALSE,
                 infomap_version = c("2.1.0", 
                                     "2.6.0",
                                     "2.7.1",
                                     "2.8.0"))

ERROR: Error in install_binaries(binpath = "tempdir", download_only = FALSE, : could not find function "install_binaries"


In [11]:
## Steps
### Load Data
#OGR <- st_read(file_OGR)
cat(capture.output(ls.str()), sep = "\n", file = paste0(out_dir, Sys.Date(), "_bioregion_init_param.txt")) #Get initial parameters
#### Species data v. tabular
tab_sp <- read.table(file_sp, header = TRUE, sep = "\t", dec = ".")
tab_sp <- tab_sp[tab_sp[, col_sp] != rm_sp, ]

# #### Localities data v. xlsx
# tab_loc <- read.xlsx(file_sp, "Localities", na.strings = "")
# #Set true NA values
# tab_loc[is.na(tab_loc)] <- ""
# tab_loc[apply(tab_loc, c(1, 2), function(x) grepl("NA", x))] <- NA

# ###Samples data
# tab_samp <- read.xlsx(file_sp, "Samples", na.strings = "")
# #Set true NA values
# tab_samp[is.na(tab_samp)] <- ""
# tab_samp[apply(tab_samp, c(1, 2), function(x) grepl("NA", x))] <- NA
# 
# ###Species data
# tab_sp <- read.xlsx(file_sp, "Sorting", na.strings = "")
# #Set true NA values
# tab_sp[is.na(tab_sp)] <- ""
# tab_sp[apply(tab_sp, c(1, 2), function(x) grepl("NA", x))] <- NA
# #Set loc name
# tab_sp$ID_locality <- tab_samp[match(tab_sp$ID_sample, tab_samp$ID_sample), "ID_locality"]

#Filter NA values
tab_sp_short <- na.omit(unique(tab_sp[!is.na(tab_sp[, col_sp]), c(col_loc, col_sp)]))
tab_sp_short$occ <- TRUE
# tab_sp_short[tab_sp_short[, col_sp] == 'diaphanosoma_sarsi', col_sp] <- "diaphanosoma_australiensis"

### Co-occurrence matrix

matrix_cooc <- as.matrix(
  tab_sp_short %>% 
    tidyr::pivot_wider(names_from = col_sp, 
                       values_from = "occ", 
                       values_fill =  FALSE))

matrix_co <- matrix_cooc[, -which(colnames(matrix_cooc) == col_loc)]

mat_co <- apply(matrix_co, 2, function(c) as.numeric(as.logical(c)))

rownames(mat_co) <- matrix_cooc[, col_loc]

### Create network

net <- mat_to_net(mat_co, weight = TRUE, remove_zeroes = FALSE)

### Clusterisation

# install_binaries(binpath = "tempdir" , infomap_version = c("2.7.1"))

net_infomap <- netclu_infomap(net, 
                              seed = 589, 
                              weight = TRUE,
                              bipartite = TRUE,
                              numtrials = 1000)

### Extract cluster table and format to get the right format for GDF conversion
clus_tab <- net_infomap$clusters
colnames(clus_tab) <- c("Name", paste0("lvl", 1:(length(clus_tab)-1)))

clus_tab$id[match(c(sort(colnames(mat_co)), sort(rownames(mat_co))), clus_tab$Name)] <- 1:nrow(clus_tab)

### Write in GDF to visualize on Gephi
writeGDF(db = tab_sp_short, 
         network = clus_tab, 
         site.field = col_loc, 
         species.field = col_sp,
         # color.field = "color", 
         filename = paste0(out_dir, "res_bioregion", Sys.Date(),".gdf"))

### Species table
species <- clus_tab[match(unique(tab_sp[, col_sp]), clus_tab[, "Name"]), ]
write.table(species, file = paste0(out_dir, "sp_bioreg", Sys.Date(), ".tabular"), sep = "\t", dec = ".", 
            quote = FALSE, row.names = FALSE)

### Site table
clus_tab[, l_col] <- tab_sp[match(clus_tab[, "Name"], tab_sp[, col_loc]), l_col]
site <- clus_tab[match(unique(tab_sp[, col_loc]), clus_tab[, "Name"]), ]

write.table(site, file = paste0(out_dir, "loc_bioreg", Sys.Date(), ".tabular"), sep = "\t", dec = ".", 
            quote = FALSE, row.names = FALSE)

Infomap 2.8.0 is not installed... Please have a look at https//bioRgeo.github.io/bioregion/articles/a1_install_binary_files.html for more details.
It should be located in /tmp/RtmpA8BWxT/bin/INFOMAP/2.8.0/



ERROR: Error in `colnames<-`(`*tmp*`, value = c("Name", "lvl1", "lvl0", "lvl-1": attempt to set 'colnames' on an object with less than two dimensions
